In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn requests pyyaml

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_absolute_error
import warnings
import os
import requests
import zipfile
import yaml
import glob
from datetime import datetime

warnings.filterwarnings("ignore")
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*70)
print("SPORTS BETTING SYSTEM - DEMO VERSION")
print("="*70)
print("Features: Data Loading, ML Training, Visualizations, Betting Recommendations")
print("Sports: Football, Tennis, Cricket")
print("="*70)



In [ ]:
# ============================================================================
# DATA LOADING FUNCTIONS
# ============================================================================

def download_football_data(year):
    """Download football data from online source"""
    try:
        url = f"https://www.football-data.co.uk/mmz4281/{str(year)[2:]}{str(year+1)[2:]}/E0.csv"
        print(f"Downloading football data from: {url}")
        df = pd.read_csv(url)
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        print(f"Downloaded {len(df)} football matches for {year}")
        return df
    except Exception as e:
        print(f"Download failed: {e}")
        return pd.DataFrame()

def download_tennis_data(year):
    """Download tennis data (synthetic if real data unavailable)"""
    try:
        # Try to download real data
        url = f"https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/atp_matches_{year}.csv"
        print(f"Downloading tennis data from: {url}")
        df = pd.read_csv(url)
        print(f"Downloaded {len(df)} tennis matches for {year}")
        return df
    except:
        print("Using synthetic tennis data")
        np.random.seed(42 + year)
        players = ['Djokovic', 'Nadal', 'Federer', 'Murray', 'Zverev', 'Medvedev']
        n_matches = 100
        
        data = {
            'winner_name': np.random.choice(players, n_matches),
            'loser_name': np.random.choice(players, n_matches),
            'winner_rank': np.random.randint(1, 50, n_matches),
            'loser_rank': np.random.randint(1, 50, n_matches),
            'surface': np.random.choice(['Hard', 'Clay', 'Grass'], n_matches),
        }
        
        for i in range(len(data['winner_name'])):
            while data['winner_name'][i] == data['loser_name'][i]:
                data['loser_name'][i] = np.random.choice(players)
        
        return pd.DataFrame(data)

def download_cricket_data(year):
    """Download cricket data - with improved error handling and fallback"""
    cricket_dir = f"data/cricket"
    os.makedirs(cricket_dir, exist_ok=True)
    
    # First, try to use existing CSV if available
    csv_file = f"data/cricket_{year}.csv"
    if os.path.exists(csv_file):
        print(f"Found existing cricket CSV: {csv_file}")
        try:
            df = pd.read_csv(csv_file)
            if not df.empty:
                print(f"Loaded {len(df)} cricket matches from CSV")
                return df
        except:
            pass
    
    # Try to download and process YAML files
    try:
        url = f"https://cricsheet.org/downloads/all_json.zip"
        zip_path = os.path.join(cricket_dir, "all_json.zip")
        
        if not os.path.exists(zip_path):
            print(f"Downloading cricket data from cricsheet.org...")
            try:
                response = requests.get(url, timeout=120)
                response.raise_for_status()
                with open(zip_path, 'wb') as f:
                    f.write(response.content)
                print(f"Downloaded cricket data zip file")
            except Exception as e:
                print(f"Download failed: {e}")
                raise
        
        # Extract if needed
        yaml_files = glob.glob(os.path.join(cricket_dir, "*.yaml"))
        if not yaml_files:
            print(f"Extracting cricket data...")
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(cricket_dir)
            yaml_files = glob.glob(os.path.join(cricket_dir, "*.yaml"))
        
        # Process YAML files
        if yaml_files:
            print(f"Found {len(yaml_files)} YAML files, processing...")
            all_rows = []
            processed = 0
            
            for yaml_file in yaml_files[:100]:  # Limit for demo
                try:
                    with open(yaml_file, 'r', encoding='utf-8') as f:
                        data = yaml.safe_load(f)
                    
                    if not data:
                        continue
                    
                    info = data.get("info", {})
                    teams = info.get("teams", [])
                    outcome = data.get("outcome", {})
                    
                    if len(teams) >= 2:
                        team1_runs = 0
                        team2_runs = 0
                        
                        # Process innings
                        innings_list = data.get("innings", [])
                        for i, innings in enumerate(innings_list):
                            if isinstance(innings, dict):
                                for inning_name, inning_data in innings.items():
                                    if isinstance(inning_data, dict):
                                        # Calculate runs from deliveries
                                        total_runs = 0
                                        overs = inning_data.get("overs", [])
                                        for over in overs:
                                            if isinstance(over, dict):
                                                deliveries = over.get("deliveries", [])
                                                for delivery in deliveries:
                                                    if isinstance(delivery, dict):
                                                        for ball, ball_data in delivery.items():
                                                            if isinstance(ball_data, dict):
                                                                runs_data = ball_data.get("runs", {})
                                                                if isinstance(runs_data, dict):
                                                                    total_runs += runs_data.get("total", 0)
                                        
                                        if "1st" in inning_name.lower() or i == 0:
                                            team1_runs = total_runs
                                        else:
                                            team2_runs = total_runs
                        
                        # Only add if we have valid runs data
                        if team1_runs > 0 or team2_runs > 0:
                            all_rows.append({
                                'team1': teams[0],
                                'team2': teams[1],
                                'team1_runs': team1_runs,
                                'team2_runs': team2_runs,
                                'winner': outcome.get("winner", teams[0] if team1_runs > team2_runs else teams[1]),
                                'match_date': info.get("dates", [None])[0] if info.get("dates") else None
                            })
                            processed += 1
                except Exception as e:
                    continue
            
            if all_rows:
                df = pd.DataFrame(all_rows)
                print(f"Processed {len(df)} cricket matches from YAML files")
                # Save to CSV for future use
                df.to_csv(csv_file, index=False)
                return df
            else:
                print(f"No valid cricket data extracted from YAML files")
                raise Exception("No data extracted")
        else:
            print(f"No YAML files found")
            raise Exception("No YAML files")
            
    except Exception as e:
        print(f"Cricket data download/processing failed: {e}")
        print(f"Generating synthetic cricket data for demonstration...")
    
    # Generate synthetic cricket data (always works)
    np.random.seed(42 + year)
    n_matches = 100
    
    teams_pool = ['India', 'Australia', 'England', 'Pakistan', 'South Africa', 
                  'New Zealand', 'West Indies', 'Sri Lanka', 'Bangladesh', 'Afghanistan']
    
    synthetic_data = []
    for i in range(n_matches):
        team1 = np.random.choice(teams_pool)
        team2 = np.random.choice([t for t in teams_pool if t != team1])
        
        team1_runs = np.random.randint(200, 350)
        team2_runs = np.random.randint(200, 350)
        
        # Ensure winner matches runs
        if team1_runs > team2_runs:
            winner = team1
        elif team2_runs > team1_runs:
            winner = team2
        else:
            winner = np.random.choice([team1, team2])
        
        synthetic_data.append({
            'team1': team1,
            'team2': team2,
            'team1_runs': team1_runs,
            'team2_runs': team2_runs,
            'winner': winner,
            'match_date': f"{year}-{np.random.randint(1,13):02d}-{np.random.randint(1,29):02d}"
        })
    
    df = pd.DataFrame(synthetic_data)
    print(f"Generated {len(df)} synthetic cricket matches for {year}")
    
    # Save synthetic data to CSV
    df.to_csv(csv_file, index=False)
    print(f"Saved synthetic cricket data to {csv_file}")
    
    return df

def load_sport_data(sport, year, use_cache=True):
    """Load sport data - check local CSV first, then download"""
    os.makedirs("data", exist_ok=True)
    csv_file = f"data/{sport.lower()}_{year}.csv"
    
    # Check if CSV exists locally
    if use_cache and os.path.exists(csv_file):
        print(f"Loading {sport} {year} from local CSV...")
        df = pd.read_csv(csv_file)
        print(f"Loaded {len(df)} records from {csv_file}")
        return df
    
    # Download data
    print(f"Fetching {sport} data for {year}...")
    
    if sport == "Football":
        df = download_football_data(year)
    elif sport == "Tennis":
        df = download_tennis_data(year)
    elif sport == "Cricket":
        df = download_cricket_data(year)
    else:
        return pd.DataFrame()
    
    # Save to CSV
    if not df.empty:
        df.to_csv(csv_file, index=False)
        print(f"Saved to {csv_file}")
    
    return df

print("Data loading functions ready!")



In [ ]:
# ============================================================================
# FEATURE ENGINEERING
# ============================================================================

def add_features(df, sport):
    """Add features for ML model"""
    if sport == "Football":
        if 'FTHG' in df.columns and 'FTAG' in df.columns:
            df['HomeGoals'] = df['FTHG']
            df['AwayGoals'] = df['FTAG']
            df['TotalGoals'] = df['HomeGoals'] + df['AwayGoals']
            df['GoalDiff'] = df['HomeGoals'] - df['AwayGoals']
            df['Result'] = df['FTR']  # H, D, A
        else:
            # Synthetic features
            df['HomeGoals'] = np.random.randint(0, 5, len(df))
            df['AwayGoals'] = np.random.randint(0, 5, len(df))
            df['TotalGoals'] = df['HomeGoals'] + df['AwayGoals']
            df['GoalDiff'] = df['HomeGoals'] - df['AwayGoals']
            df['Result'] = np.random.choice(['H', 'D', 'A'], len(df), p=[0.45, 0.25, 0.30])
        
        features = ['HomeGoals', 'AwayGoals', 'TotalGoals', 'GoalDiff']
        target = 'Result'
        le = LabelEncoder()
        df['ResultEnc'] = le.fit_transform(df['Result'])
        return df, features, target, le
    
    elif sport == "Tennis":
        df['RankDiff'] = df['winner_rank'] - df['loser_rank']
        df['Upset'] = (df['RankDiff'] > 0).astype(int)  # Lower rank wins
        df['SurfaceAdvantage'] = np.random.randint(0, 2, len(df))  # Placeholder
        
        features = ['winner_rank', 'loser_rank', 'RankDiff', 'Upset', 'SurfaceAdvantage']
        target = 'Upset'
        return df, features, target, None
    
    elif sport == "Cricket":
        df['RunDiff'] = df['team1_runs'] - df['team2_runs']
        df['TotalRuns'] = df['team1_runs'] + df['team2_runs']
        df['HighScoring'] = (df['TotalRuns'] > 500).astype(int)
        df['winner'] = (df['RunDiff'] > 0).astype(int)  # 1 if team1 wins
        
        features = ['team1_runs', 'team2_runs', 'RunDiff', 'TotalRuns', 'HighScoring']
        target = 'winner'
        return df, features, target, None
    
    return df, [], None, None

print("Feature engineering functions ready!")



In [ ]:
# ============================================================================
# ML MODEL TRAINING
# ============================================================================

def train_model(df, features, target, sport):
    """Train ML model for sport prediction"""
    print(f"\nTraining {sport} model...")
    
    X = df[features].fillna(0)
    y = df[target]
    
    if len(X) < 10:
        print(f"Insufficient data for {sport}")
        return None, 0, None
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    if sport in ["Football", "Tennis"]:
        # Classification
        model = RandomForestClassifier(n_estimators=100, random_state=42)
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        
        print(f"{sport} Model Accuracy: {accuracy:.3f}")
        print(f"   Training samples: {len(X_train)}, Test samples: {len(X_test)}")
        
        return model, accuracy, None
    
    elif sport == "Cricket":
        # Regression
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        
        print(f"{sport} Model MAE: {mae:.3f}")
        print(f"   Training samples: {len(X_train)}, Test samples: {len(X_test)}")
        
        return model, mae, None
    
    return None, 0, None

print("ML training functions ready!")



In [ ]:
def create_visualizations(df, sport, year, model, features, target):
    """Create simple visualizations for the sport"""
    print(f"\nCreating visualizations for {sport} {year}...")
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(f'{sport} {year} - Analysis & Predictions', fontsize=16, fontweight='bold')
    
    # Plot 1: Distribution of target variable
    ax1 = axes[0, 0]
    if sport == "Football":
        df[target].value_counts().plot(kind='bar', ax=ax1, color=['green', 'blue', 'red'])
        ax1.set_title('Match Results Distribution', fontsize=12, fontweight='bold')
        ax1.set_xlabel('Result (H=Home, D=Draw, A=Away)')
        ax1.set_ylabel('Count')
    elif sport == "Tennis":
        df['winner_name'].value_counts().head(10).plot(kind='bar', ax=ax1)
        ax1.set_title('Top 10 Winners', fontsize=12, fontweight='bold')
        ax1.set_xlabel('Player')
        ax1.set_ylabel('Wins')
        plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
    elif sport == "Cricket":
        # Show distribution with winner information
        if 'winner' in df.columns:
            team1_wins = df[df['winner'] == 1]['TotalRuns']
            team2_wins = df[df['winner'] == 0]['TotalRuns']
            
            ax1.hist([team1_wins, team2_wins], bins=20, edgecolor='black', 
                    alpha=0.7, label=['Team 1 Wins', 'Team 2 Wins'], color=['blue', 'orange'])
            ax1.legend()
            ax1.set_title('Total Runs Distribution by Winner', fontsize=12, fontweight='bold')
        else:
            df['TotalRuns'].hist(bins=20, ax=ax1, edgecolor='black')
            ax1.set_title('Total Runs Distribution', fontsize=12, fontweight='bold')
        ax1.set_xlabel('Total Runs')
        ax1.set_ylabel('Frequency')
    
    # Plot 2: Feature importance
    ax2 = axes[0, 1]
    if hasattr(model, 'feature_importances_'):
        importance = model.feature_importances_
        indices = np.argsort(importance)[::-1]
        ax2.barh(range(len(importance)), importance[indices])
        ax2.set_yticks(range(len(importance)))
        ax2.set_yticklabels([features[i] for i in indices])
        ax2.set_title('Feature Importance', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Importance')
    else:
        ax2.text(0.5, 0.5, 'Feature importance\nnot available', 
                ha='center', va='center', transform=ax2.transAxes, fontsize=12)
        ax2.set_title('Feature Importance', fontsize=12, fontweight='bold')
    
    # Plot 3: Prediction confidence
    ax3 = axes[1, 0]
    if sport in ["Football", "Tennis"]:
        X = df[features].fillna(0)
        if hasattr(model, 'predict_proba'):
            probabilities = model.predict_proba(X)
            max_probs = np.max(probabilities, axis=1)
            ax3.hist(max_probs, bins=20, edgecolor='black', alpha=0.7)
            ax3.axvline(x=0.6, color='red', linestyle='--', label='Betting Threshold (60%)', linewidth=2)
            ax3.set_title('Prediction Confidence Distribution', fontsize=12, fontweight='bold')
            ax3.set_xlabel('Confidence Score')
            ax3.set_ylabel('Frequency')
            ax3.legend()
    elif sport == "Cricket":
        X = df[features].fillna(0)
        predictions = model.predict(X)
        
        # Calculate confidence from prediction variance (same as other sports format)
        if hasattr(model, 'estimators_'):
            tree_preds = np.array([tree.predict(X) for tree in model.estimators_])
            pred_std = np.std(tree_preds, axis=0)
            confidence_scores = 1 / (1 + pred_std * 2)
            confidence_scores = np.clip(confidence_scores, 0.3, 0.95)
        else:
            confidence_scores = np.ones(len(predictions)) * 0.6
        
        # Show confidence distribution histogram (same format as Football/Tennis)
        ax3.hist(confidence_scores, bins=20, edgecolor='black', alpha=0.7)
        ax3.axvline(x=0.6, color='red', linestyle='--', label='Betting Threshold (60%)', linewidth=2)
        ax3.set_title('Prediction Confidence Distribution', fontsize=12, fontweight='bold')
        ax3.set_xlabel('Confidence Score')
        ax3.set_ylabel('Frequency')
        ax3.legend()
    
    # Plot 4: Additional analysis
    ax4 = axes[1, 1]
    if sport == "Football":
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
            monthly_goals = df.groupby(df['Date'].dt.to_period('M'))['TotalGoals'].mean()
            monthly_goals.plot(kind='line', ax=ax4, marker='o', linewidth=2)
            ax4.set_title('Average Goals per Month', fontsize=12, fontweight='bold')
            ax4.set_xlabel('Month')
            ax4.set_ylabel('Avg Goals')
            plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')
        else:
            df['TotalGoals'].hist(bins=20, ax=ax4, edgecolor='black')
            ax4.set_title('Total Goals Distribution', fontsize=12, fontweight='bold')
    elif sport == "Tennis":
        df['RankDiff'].hist(bins=20, ax=ax4, edgecolor='black')
        ax4.set_title('Rank Difference Distribution', fontsize=12, fontweight='bold')
        ax4.set_xlabel('Rank Difference')
        ax4.set_ylabel('Frequency')
    elif sport == "Cricket":
        # Show predictions analysis (similar format to other sports)
        X = df[features].fillna(0)
        predictions = model.predict(X)
        
        # Calculate confidence scores
        if hasattr(model, 'estimators_'):
            tree_preds = np.array([tree.predict(X) for tree in model.estimators_])
            pred_std = np.std(tree_preds, axis=0)
            confidence_scores = 1 / (1 + pred_std * 2)
            confidence_scores = np.clip(confidence_scores, 0.3, 0.95)
        else:
            confidence_scores = np.ones(len(predictions)) * 0.6
        
        # Scatter plot: Predictions vs Total Runs with confidence colors
        scatter = ax4.scatter(df['TotalRuns'], predictions, c=confidence_scores, 
                             cmap='RdYlGn', alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
        ax4.axhline(y=0.5, color='red', linestyle='--', linewidth=2, label='Decision Threshold (0.5)')
        ax4.set_title('Predictions: Winner Probability vs Total Runs', fontsize=12, fontweight='bold')
        ax4.set_xlabel('Total Runs in Match')
        ax4.set_ylabel('Predicted Winner Probability\n(>0.5 = Team 1, <0.5 = Team 2)')
        ax4.legend()
        plt.colorbar(scatter, ax=ax4, label='Confidence Score')
    
    plt.tight_layout()
    
    # Save figure
    os.makedirs("reports", exist_ok=True)
    fig_path = f"reports/{sport}_{year}_analysis.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"Saved visualization: {fig_path}")
    
    plt.show()
    
    return fig_path

print("Visualization functions ready!")



In [ ]:
def generate_betting_recommendations(df, model, features, sport, confidence_threshold=0.6):
    """Generate betting recommendations based on ML predictions"""
    print(f"\nGenerating betting recommendations for {sport}...")
    
    X = df[features].fillna(0)
    recommendations = []
    
    if sport in ["Football", "Tennis"]:
        if hasattr(model, 'predict_proba'):
            probabilities = model.predict_proba(X)
            predictions = model.predict(X)
            max_probs = np.max(probabilities, axis=1)
            
            for i, (pred, conf) in enumerate(zip(predictions, max_probs)):
                if conf >= confidence_threshold:
                    if sport == "Football":
                        outcome_map = {0: "Away Win", 1: "Draw", 2: "Home Win"}
                        selection = outcome_map.get(pred, "Unknown")
                    else:
                        selection = "Player 1 Win" if pred == 0 else "Player 2 Win"
                    
                    odds = 1 / conf * 1.1  # Add 10% margin
                    stake = 10 * conf  # Base stake * confidence
                    
                    recommendations.append({
                        'match_id': f'{sport.lower()}_match_{i}',
                        'selection': selection,
                        'confidence': round(conf, 3),
                        'odds': round(odds, 2),
                        'stake': round(stake, 2),
                        'potential_return': round(stake * odds, 2)
                    })
    
    elif sport == "Cricket":
        predictions = model.predict(X)
        # Calculate confidence from prediction variance
        if hasattr(model, 'estimators_'):
            tree_preds = np.array([tree.predict(X) for tree in model.estimators_])
            pred_std = np.std(tree_preds, axis=0)
            confidence_scores = 1 / (1 + pred_std * 2)
            confidence_scores = np.clip(confidence_scores, 0.3, 0.95)
        else:
            confidence_scores = np.ones(len(predictions)) * 0.6
        
        for i, (pred, conf) in enumerate(zip(predictions, confidence_scores)):
            if conf >= confidence_threshold:
                selection = "Team 1 Win" if pred > 0.5 else "Team 2 Win"
                odds = 1 / conf * 1.1
                stake = 10 * conf
                
                recommendations.append({
                    'match_id': f'cricket_match_{i}',
                    'selection': selection,
                    'confidence': round(conf, 3),
                    'odds': round(odds, 2),
                    'stake': round(stake, 2),
                    'potential_return': round(stake * odds, 2)
                })
    
    print(f"Generated {len(recommendations)} betting recommendations")
    if recommendations:
        print("\nTop 5 Recommendations:")
        for i, rec in enumerate(recommendations[:5], 1):
            print(f"  {i}. {rec['selection']} @ {rec['odds']} (Confidence: {rec['confidence']:.1%}, Stake: ${rec['stake']:.2f})")
    
    return recommendations

print("Betting recommendation functions ready!")



In [ ]:
print("\n" + "="*70)
print("STARTING SPORTS BETTING SYSTEM DEMO")
print("="*70)

sports = ["Football", "Tennis", "Cricket"]
year = 2021

results = {}

for sport in sports:
    print(f"\n{'='*70}")
    print(f"Processing {sport} {year}")
    print(f"{'='*70}")
    
    # Load data
    df = load_sport_data(sport, year, use_cache=True)
    
    if df.empty:
        print(f"No data available for {sport} {year}")
        continue
    
    # Feature engineering
    df, features, target, le = add_features(df, sport)
    
    if not features:
        print(f"Could not create features for {sport}")
        continue
    
    # Train model
    model, metric, _ = train_model(df, features, target, sport)
    
    if model is None:
        continue
    
    # Create visualizations
    fig_path = create_visualizations(df, sport, year, model, features, target)
    
    # Generate betting recommendations
    recommendations = generate_betting_recommendations(df, model, features, sport)
    
    results[sport] = {
        'data_points': len(df),
        'metric': metric,
        'recommendations': len(recommendations),
        'figure': fig_path
    }

# Summary
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)
for sport, result in results.items():
    print(f"\n{sport}:")
    print(f"  Data Points: {result['data_points']}")
    print(f"  Model Metric: {result['metric']:.3f}")
    print(f"  Betting Recommendations: {result['recommendations']}")
    print(f"  Visualization: {result['figure']}")

print("\n" + "="*70)
print("="*70)

